In [1]:
import os, re, warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_DIR = "Code Outputs/Climate EDA Outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
def out(n): return os.path.join(OUTPUT_DIR, n)

# Loading
clim = pd.read_excel("Code Outputs/Climate Data Extraction Outputs/Lake_Climate_Monthly.xlsx"); clim["Date"] = pd.to_datetime(clim["Date"])
lev  = pd.read_excel("Code Outputs/Gap Interpolation Outputs/Unified_Interpolated_Levels.xlsx"); lev["Date"] = pd.to_datetime(lev["Date"])
LAKES = sorted(clim["Reservoir"].unique())

def deseasonalise(s):
    """Subtract the calendar-month climatology"""
    return s - s.groupby(s.index.month).transform("mean")

def lead_corr(target, driver, max_lag=12):
    """corr(target[t], driver[t-k]) for k=0..max_lag ; k>0 => driver leads."""
    j = target.dropna().index.intersection(driver.dropna().index)
    T, D = target.loc[j], driver.loc[j]
    return {k: T.corr(D.shift(k)) for k in range(0, max_lag + 1)}

# level (wide) and its monthly change, deseasonalised
level_w = lev.pivot(index="Date", columns="Reservoir", values="Level_m").sort_index().asfreq("MS")
dlevel  = level_w.diff()                                   # the flux
dlevel_anom = dlevel.apply(deseasonalise)                  # deseasonalised change

# Optional climate indices
def parse_index(path):
    if not os.path.exists(path): return None
    recs = []
    for line in open(path):
        toks = re.split(r"[,\s]+", line.strip())
        if not toks or toks == [""]: continue
        m = re.match(r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})$", toks[0])
        if m and len(toks) >= 2:
            try: recs.append((pd.Timestamp(int(m[1]), int(m[2]), 1), float(toks[1])))
            except ValueError: pass
            continue
        if len(toks) == 13 and re.fullmatch(r"\d{4}", toks[0]):
            try: vals = [float(t) for t in toks[1:]]
            except ValueError: continue
            for mo, v in enumerate(vals, 1):
                recs.append((pd.Timestamp(int(toks[0]), mo, 1), v))
    if not recs: return None
    s = pd.Series(dict(recs)).sort_index(); s[s < -90] = np.nan
    return s.asfreq("MS")

dmi  = parse_index("Climate Indices/dmi.csv")  if os.path.exists("Climate Indices/dmi.csv")  else parse_index("dmi.data")
nino = parse_index("Climate Indices/nino34.csv") if os.path.exists("Climate Indices/nino34.csv") else parse_index("nino34.data")
if nino is not None: nino = deseasonalise(nino)

# Lead-lag vs delta level
rows = []
fig, axes = plt.subplots(4, 2, figsize=(15, 16)); axes = axes.flatten()
for i, lk in enumerate(LAKES):
    g = clim[clim.Reservoir == lk].set_index("Date").sort_index()
    precip_anom = deseasonalise(g["precip_mm"].asfreq("MS"))
    wb_anom     = deseasonalise(g["water_balance_mm"].asfreq("MS"))
    dl = dlevel_anom[lk]

    lc_p  = lead_corr(dl, precip_anom)
    lc_wb = lead_corr(dl, wb_anom)
    row = {"Lake": lk,
           "precip_lag0": round(lc_p[0], 3),
           "precip_peak_lag": max(lc_p, key=lambda k: abs(lc_p[k])),
           "precip_peak_corr": round(max(lc_p.values(), key=abs), 3),
           "wb_lag0": round(lc_wb[0], 3),
           "wb_peak_lag": max(lc_wb, key=lambda k: abs(lc_wb[k])),
           "wb_peak_corr": round(max(lc_wb.values(), key=abs), 3)}
    # indices vs delta-level (if available)
    for nm, series in [("DMI", dmi), ("Nino34", nino)]:
        if series is not None:
            lc = lead_corr(dl, series)
            row[f"{nm}_peak_lag"]  = max(lc, key=lambda k: abs(lc[k]))
            row[f"{nm}_peak_corr"] = round(max(lc.values(), key=abs), 3)
    rows.append(row)

    ax = axes[i]
    ax.plot(list(lc_p),  list(lc_p.values()),  marker="o", label="precip")
    ax.plot(list(lc_wb), list(lc_wb.values()), marker="s", label="water balance")
    ax.axhline(0, color="k", lw=.8); ax.set_title(lk, fontweight="bold")
    ax.set_xlabel("driver leads dLevel by k months"); ax.set_ylabel("corr")
    if i == 0: ax.legend(fontsize=8)
axes[7].set_visible(False)
plt.suptitle("Lagged Correlation: climate driver vs MONTHLY LEVEL CHANGE (deseasonalised)",
             fontweight="bold", y=.995)
plt.tight_layout(); plt.savefig(out("CLIM_06_delta_lagged.png"), dpi=800); plt.close()

summary = pd.DataFrame(rows)
summary.to_csv(out("CLIM_06_delta_leadlag.csv"), index=False)
print("=== DRIVERS vs DELTA-LEVEL (deseasonalised) ===")
print(summary.to_string(index=False))

=== DRIVERS vs DELTA-LEVEL (deseasonalised) ===
           Lake  precip_lag0  precip_peak_lag  precip_peak_corr  wb_lag0  wb_peak_lag  wb_peak_corr  DMI_peak_lag  DMI_peak_corr  Nino34_peak_lag  Nino34_peak_corr
    Lake Albert        0.248                1             0.349    0.260            1         0.351             3          0.269                5             0.212
    Lake Edward        0.283                0             0.283    0.293            0         0.293             6          0.122               12            -0.154
      Lake Kivu        0.251                1             0.329    0.260            1         0.332             5          0.148                1             0.151
    Lake Malawi        0.386                1             0.551    0.382            1         0.536             3          0.184               11            -0.144
Lake Tanganyika        0.357                1             0.556    0.355            1         0.542             4          0.364    